In [0]:
from __future__ import annotations

import sys
from pathlib import Path

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "common").exists():
        repository_root = str(candidate)
        if repository_root not in sys.path:
            sys.path.insert(0, repository_root)
        break

from notebooks.common.audit import add_audit_columns
from notebooks.common.metrics import (
    calculate_data_quality_score,
    create_pipeline_metric_df,
    print_summary,
)
from notebooks.common.validation import (
    apply_validation_rules,
    split_valid_and_quarantine,
)
from notebooks.config.northstar.paths import PATHS
from notebooks.config.northstar.schemas import ELIGIBILITY_SCHEMA
from notebooks.config.northstar.validation_rules import (
    build_eligibility_validation_rules,
)

In [0]:
NOTEBOOK_VERSION = "1.1.0"
PIPELINE_NAME = "northstar_bronze_to_silver_eligibility"

BUSINESS_DATE = "2026-07-01"
BATCH_ID = "northstar-eligibility-20260702"
SOURCE_SYSTEM = "Eligibility Platform"
WRITE_MODE = "overwrite"
BRONZE_FILENAME = "eligibility_20260702.csv"

BRONZE_PATH = PATHS.bronze_file("eligibility", BRONZE_FILENAME)
SILVER_PATH = PATHS.silver_path("eligibility")
QUARANTINE_PATH = PATHS.quarantine_path("eligibility")
METRICS_PATH = PATHS.gold_path("data_quality_metrics")

print(f"Bronze path: {BRONZE_PATH}")
print(f"Silver path: {SILVER_PATH}")

In [0]:
def read_bronze_eligibility(path: str) -> DataFrame:
    bronze_schema = StructType(
        [
            field
            for field in ELIGIBILITY_SCHEMA.fields
            if field.name != "_corrupt_record"
        ]
        + [StructField("_corrupt_record", StringType(), nullable=True)]
    )

    return (
        spark.read.format("csv")
        .schema(bronze_schema)
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .option("dateFormat", "yyyy-MM-dd")
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ssX")
        .load(path)
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file_path"),
        )
    )


bronze_df = read_bronze_eligibility(BRONZE_PATH)

records_read = bronze_df.count()

print(f"Eligibility Bronze records read: {records_read:,}")

display(bronze_df.limit(10))

In [0]:
validated_df = apply_validation_rules(
    bronze_df,
    build_eligibility_validation_rules(as_of_date=BUSINESS_DATE),
)

valid_df, quarantine_df = split_valid_and_quarantine(validated_df)

valid_count = valid_df.count()
quarantine_count = quarantine_df.count()

print(f"Records read: {records_read:,}")
print(f"Valid rows: {valid_count:,}")
print(f"Rejected rows: {quarantine_count:,}")
print("Reconciled:", records_read == valid_count + quarantine_count)

display(quarantine_df.limit(10))

In [0]:
silver_df = add_audit_columns(
    valid_df,
    batch_id=BATCH_ID,
    source_system=SOURCE_SYSTEM,
)

quarantine_output_df = add_audit_columns(
    quarantine_df,
    batch_id=BATCH_ID,
    source_system=SOURCE_SYSTEM,
)

display(silver_df.limit(10))

In [0]:
(
    silver_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

silver_written_count = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
    .count()
)

if silver_written_count != valid_count:
    raise RuntimeError(
        f"Silver validation failed: "
        f"expected={valid_count:,}, "
        f"actual={silver_written_count:,}"
    )

print(f"Silver rows written: {silver_written_count:,}")

In [0]:
(
    quarantine_output_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(QUARANTINE_PATH)
)

quarantine_written_count = (
    spark.read
    .format("delta")
    .load(QUARANTINE_PATH)
    .count()
)

print(f"Quarantine rows written: {quarantine_written_count:,}")

In [0]:
data_quality_score = calculate_data_quality_score(
    records_read,
    quarantine_count,
)

print_summary(
    records_read=records_read,
    records_written=silver_written_count,
    rejected_records=quarantine_written_count,
    data_quality_score=data_quality_score,
)

In [0]:
metric_df = create_pipeline_metric_df(
    spark,
    pipeline_name=PIPELINE_NAME,
    notebook_version=NOTEBOOK_VERSION,
    batch_id=BATCH_ID,
    business_date=BUSINESS_DATE,
    source_system=SOURCE_SYSTEM,
    source_path=BRONZE_PATH,
    target_path=SILVER_PATH,
    records_read=records_read,
    records_written=silver_written_count,
    records_rejected=quarantine_written_count,
    run_status="SUCCEEDED",
)

(
    metric_df.write
    .format("delta")
    .mode("append")
    .save(METRICS_PATH)
)

print("\nEligibility Bronze-to-Silver processing completed successfully.")
print(f"Notebook version: {NOTEBOOK_VERSION}")
print(f"Silver rows:      {silver_written_count:,}")
print(f"Quarantine rows:  {quarantine_written_count:,}")
print(f"Quality score:    {data_quality_score:.2f}%")
print(f"Metrics path:     {METRICS_PATH}")